In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from loguru import logger

pd.set_option("display.max_columns", None)

In [2]:
!ls ../data/datasets/

dataset_level_01_daily_total.parquet
dataset_level_02_daily_state.parquet
dataset_level_03_daily_cat.parquet
dataset_level_04_daily_dept.parquet
dataset_level_05_daily_state_cat.parquet
dataset_level_06_daily_store.parquet
dataset_level_07_daily_state_dept.parquet
dataset_level_08_daily_store_cat.parquet
dataset_level_09_daily_store_dept.parquet
dataset_level_10_weekly_item.parquet
dataset_level_11_weekly_item_state.parquet
dataset_level_12_weekly_item_store.parquet


# Data

In [3]:
LEVEL = 'level_10_weekly_item'

# Granularidad temporal del nivel ('daily' o 'weekly'), embebida en el nombre
# (level_{id}_{grain}_{name}); se usa más abajo para adaptar estacionalidades
# (seasonal naive, SARIMA/ETS/Theta/TBATS/Prophet) al período correcto.
GRAIN = 'weekly' if '_weekly_' in LEVEL else 'daily'
SEASON_LENGTH = {"daily": 7, "weekly": 52}[GRAIN]

In [4]:
df = pd.read_parquet(f'../data/datasets/dataset_{LEVEL}.parquet')

df = df.sort_values("date").copy()

df.head()

,agg_id,item_id,dept_id,cat_id,store_id,state_id,date,sales,gross_sales,avg_sell_price,price_lag_1,price_change,price_vs_mean,has_event,has_event_2,snap,event_name_1,event_type_1,event_name_2,event_type_2,year,month,day,dayofweek,weekofyear,is_weekend,cum4,cum8,cum12,cum16,cum20,cum24,cum28,cum32,cum36,cum40,cum44,cum48,cum52,price_vs_max,days_since_release,days_since_event,days_to_event,days_to_closure,event_type_1_enc,event_type_2_enc,quarter,is_month_start,is_month_end,is_store_closed,dow_sin,dow_cos,month_sin,month_cos,doy_sin,doy_cos,price_volatility,days_since_thanksgiving,days_to_thanksgiving,days_since_newyear,days_to_newyear,zero_streak,pct_zero_28,pct_zero_90,adi_expanding,cv2_90,is_likely_stockout,lag1,lag2,lag3,lag4,lag8,lag13,lag17,lag22,lag26,lag39,lag52,rolling_mean_lag1_window_size4,rolling_std_lag1_window_size4,rolling_min_lag1_window_size4,rolling_max_lag1_window_size4,rolling_mean_lag1_window_size13,rolling_std_lag1_window_size13,rolling_min_lag1_window_size13,rolling_max_lag1_window_size13,rolling_mean_lag4_window_size4,rolling_std_lag4_window_size4,rolling_min_lag4_window_size4,rolling_max_lag4_window_size4,rolling_mean_lag4_window_size13,rolling_std_lag4_window_size13,rolling_min_lag4_window_size13,rolling_max_lag4_window_size13,rolling_mean_lag4_window_size26,rolling_std_lag4_window_size26,rolling_min_lag4_window_size26,rolling_max_lag4_window_size26,rolling_mean_lag4_window_size4_truediv_rolling_mean_lag4_window_size13,rolling_mean_lag13_window_size13,rolling_std_lag13_window_size13,rolling_min_lag13_window_size13,rolling_max_lag13_window_size13,rolling_mean_lag13_window_size26,rolling_std_lag13_window_size26,rolling_min_lag13_window_size26,rolling_max_lag13_window_size26,expanding_mean_lag13,rolling_mean_lag52_window_size4,rolling_std_lag52_window_size4,rolling_min_lag52_window_size4,rolling_max_lag52_window_size4,rolling_mean_lag52_window_size13,rolling_std_lag52_window_size13,rolling_min_lag52_window_size13,rolling_max_lag52_window_size13
0,FOODS_1_FOODS_FOODS_1_001,FOODS_1_001,FOODS_1,FOODS,TOTAL,TOTAL,2011-01-29,57.0,114.000000,2.0000,NaN,NaN,0.923397,0,0,0,NaN,NaN,NaN,NaN,2011,1,29,5,4,1,309.0,585.0,837.0,1186.0,1460.0,1705.0,1834.0,1834.0,1834.0,1944.0,2213.0,2536.0,2781.0,1.0,0,NaN,7.0,NaN,0,0,1,0,0,0,-0.974928,-0.222521,0.0,1.0,0.463258,0.886223,NaN,NaN,294.0,NaN,336.0,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
263544,FOODS_3_FOODS_FOODS_3_336,FOODS_3_336,FOODS_3,FOODS,TOTAL,TOTAL,2011-01-29,80.0,197.600006,2.4700,NaN,NaN,0.989329,0,0,0,NaN,NaN,NaN,NaN,2011,1,29,5,4,1,371.0,709.0,1142.0,1515.0,1975.0,2353.0,2807.0,3227.0,3600.0,4047.0,4364.0,4615.0,4951.0,1.0,0,NaN,7.0,NaN,0,0,1,0,0,0,-0.974928,-0.222521,0.0,1.0,0.463258,0.886223,NaN,NaN,294.0,NaN,336.0,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
582132,HOUSEHOLD_1_HOUSEHOLD_HOUSEHOLD_1_096,HOUSEHOLD_1_096,HOUSEHOLD_1,HOUSEHOLD,TOTAL,TOTAL,2011-01-29,0.0,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN,NaN,NaN,2011,1,29,5,4,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,-567,NaN,7.0,NaN,0,0,1,0,0,0,-0.974928,-0.222521,0.0,1.0,0.463258,0.886223,NaN,NaN,294.0,NaN,336.0,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
263822,FOODS_3_FOODS_FOODS_3_337,FOODS_3_337,FOODS_3,FOODS,TOTAL,TOTAL,2011-01-29,0.0,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN,NaN,NaN,2011,1,29,5,4,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,-1302,NaN,7.0,NaN,0,0,1,0,0,0,-0.974928,-0.222521,0.0,1.0,0.463258,0.886223,NaN,NaN,294.0,NaN,336.0,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,N

In [5]:
df.info()

<class 'pandas.DataFrame'>
Index: 847622 entries, 0 to 847621
Columns: 116 entries, agg_id to rolling_max_lag52_window_size13
dtypes: datetime64[ms](1), float32(80), float64(8), int16(1), int32(1), int8(15), str(10)
memory usage: 469.2 MB


# Variables

In [6]:
TARGET = "sales"

ID_COLS = ["agg_id", "date"]
CUM_COLS = [c for c in df.columns if c.startswith("cum") and c[3:].isdigit()]
LEAKY_COLS = ["gross_sales"] + CUM_COLS

FEATURES = [c for c in df.columns if c not in ID_COLS + LEAKY_COLS + [TARGET]]

# Dims estáticas candidatas: no todos los niveles tienen todas (p.ej. nivel item
# no tiene store_id/state_id), así que se filtran a las presentes en el dataset.
CATEGORICAL_FEATURES = [
    c for c in [
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2",
    ] if c in df.columns
]

NUMERICAL_FEATURES = [c for c in FEATURES if c not in CATEGORICAL_FEATURES]

FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

print(len(FEATURES))

99


# Split

In [7]:
from src.data import date_split

TEST_DAYS = 28
VALID_DAYS = 365

data_split = date_split(df, valid_days=VALID_DAYS, test_days=TEST_DAYS)

train, valid, test = data_split.train, data_split.valid, data_split.test
first_date, last_date = data_split.first_date, data_split.last_date
valid_start, test_start = data_split.valid_start, data_split.test_start

data_split.log_summary()

2026-08-22 22:15:34.336 | INFO     | src.data.split:log_summary:38 - Train : 2011-01-29 to 2015-04-24 (1,546 days, 673,829 rows)
2026-08-22 22:15:34.340 | INFO     | src.data.split:log_summary:39 - Valid : 2015-04-24 to 2016-04-23 (365 days, 158,548 rows)
2026-08-22 22:15:34.342 | INFO     | src.data.split:log_summary:40 - Test  : 2016-04-23 to 2016-05-21 (28 days, 15,245 rows)


In [8]:
from src.data import build_feature_matrices

X_train, y_train, X_valid, y_valid, X_test, y_test = build_feature_matrices(
    df, data_split, FEATURES, CATEGORICAL_FEATURES, TARGET,
)

print(f"Cantidad features: {len(FEATURES)} ({len(CATEGORICAL_FEATURES)} categoricas)")


Cantidad features: 99 (9 categoricas)


# Comparación de modelos

Antes de invertir tiempo en selección de features, se evalúan varios modelos con hiperparámetros por defecto sobre el set de validación. El objetivo es elegir la familia de modelo con mejor desempeño (WAPE/WRMSSE) para después iterar sobre ella (feature selection, tuning, etc.).

In [9]:
import time

from src.evaluation import evaluate_predictions

model_results = []
predictions_valid = {}

def evaluate_model(name, y_pred_valid, fit_time=None, category=None):
    result = evaluate_predictions(train, valid, y_valid, y_pred_valid, name, fit_time=fit_time, category=category)
    model_results.append(result)
    predictions_valid[name] = y_pred_valid
    return result

## Baseline models

In [10]:
from src.modeling import naive_last_value, seasonal_naive, drift, historical_mean, moving_average

In [11]:
t0 = time.perf_counter()
y_pred_naive = naive_last_value(train, valid, TARGET)
fit_time = time.perf_counter() - t0

evaluate_model("Naive (last value)", y_pred_naive, fit_time=fit_time, category="Naive")

2026-08-22 22:15:48.523 | INFO     | src.evaluation.metrics:evaluate_predictions:326 - Naive (last value) [Naive] | WAPE: 36.02% | WRMSSE: 1.5148 | MAE: 32.561 | RMSE: 87.024 | MAPE: 18186038425.65% | SMAPE: 48.62% | Bias: -11.02% | RMSLE: 1.1997 | TS: 48493.47 | SPEC: 6322.453 | MASE: 1.7893 | fit: 1.40s


{'model': 'Naive (last value)',
 'category': 'Naive',
 'wape': 0.36016581820414306,
 'wrmsse': 1.5148249290268527,
 'mae': 32.561331584125945,
 'rmse': 87.0235171651817,
 'mape': 181860384.25647196,
 'smape': 0.48619189713805905,
 'bias': -0.11016027186148515,
 'rmsle': 1.1997093318770862,
 'tracking_signal': 48493.47134101199,
 'spec': 6322.453276999065,
 'mase': 1.7893306142883185,
 'fit_time': 1.404642404999322}

In [12]:
t0 = time.perf_counter()
y_pred_seasonal_short = seasonal_naive(train, valid, TARGET, season_length=SEASON_LENGTH)
fit_time = time.perf_counter() - t0
evaluate_model(f"Seasonal naive ({SEASON_LENGTH}{GRAIN[0]})", y_pred_seasonal_short, fit_time=fit_time, category="Naive")

if GRAIN == "daily":
    t0 = time.perf_counter()
    y_pred_seasonal_year = seasonal_naive(train, valid, TARGET, season_length=365)
    fit_time = time.perf_counter() - t0
    evaluate_model("Seasonal naive (365d)", y_pred_seasonal_year, fit_time=fit_time, category="Naive")

2026-08-22 22:15:58.412 | INFO     | src.evaluation.metrics:evaluate_predictions:326 - Seasonal naive (52w) [Naive] | WAPE: 39.02% | WRMSSE: 1.9054 | MAE: 35.274 | RMSE: 85.365 | MAPE: 27045815904.73% | SMAPE: 56.92% | Bias: -8.75% | RMSLE: 1.4120 | TS: 35555.89 | SPEC: 7552.543 | MASE: 2.3282 | fit: 1.76s


### Drift

In [13]:
t0 = time.perf_counter()
y_pred_drift = drift(train, valid, TARGET)
fit_time = time.perf_counter() - t0

evaluate_model("Drift", y_pred_drift, fit_time=fit_time, category="Naive")

2026-08-22 22:16:08.521 | INFO     | src.evaluation.metrics:evaluate_predictions:326 -             Drift [Naive] | WAPE: 38.57% | WRMSSE: 1.5980 | MAE: 34.873 | RMSE: 92.604 | MAPE: 19788244076.88% | SMAPE: 52.07% | Bias: -8.56% | RMSLE: 1.2723 | TS: 35165.54 | SPEC: 6444.228 | MASE: 1.9155 | fit: 2.04s


{'model': 'Drift',
 'category': 'Naive',
 'wape': 0.3857323144764237,
 'wrmsse': 1.5980104863384859,
 'mae': 34.872709067744324,
 'rmse': 92.60430343554913,
 'mape': 197882440.76877788,
 'smape': 0.5207042316008177,
 'bias': -0.08555444019136753,
 'rmsle': 1.2723246896785547,
 'tracking_signal': 35165.54064668599,
 'spec': 6444.2279991850055,
 'mase': 1.9155366183335614,
 'fit_time': 2.0385253119966364}

### Historical mean

In [14]:
t0 = time.perf_counter()
y_pred_mean = historical_mean(train, valid, TARGET)
fit_time = time.perf_counter() - t0

evaluate_model("Historical mean", y_pred_mean, fit_time=fit_time, category="Naive")

2026-08-22 22:16:16.503 | INFO     | src.evaluation.metrics:evaluate_predictions:326 -   Historical mean [Naive] | WAPE: 45.72% | WRMSSE: 2.1752 | MAE: 41.336 | RMSE: 91.505 | MAPE: 30622000612.13% | SMAPE: 65.30% | Bias: -16.07% | RMSLE: 1.2111 | TS: 55710.96 | SPEC: 13445.550 | MASE: 2.9600 | fit: 0.11s


{'model': 'Historical mean',
 'category': 'Naive',
 'wape': 0.45722221359185006,
 'wrmsse': 2.1751526425763763,
 'mae': 41.33586073943823,
 'rmse': 91.50504320422341,
 'mape': 306220006.12127733,
 'smape': 0.6530253401129068,
 'bias': -0.1606597793514755,
 'rmsle': 1.2110909510123962,
 'tracking_signal': 55710.95615961511,
 'spec': 13445.549872826736,
 'mase': 2.9599513499655146,
 'fit_time': 0.11271843899885425}

### Moving average

In [15]:
MA_WINDOWS = {"daily": [7, 14, 21, 28, 35], "weekly": [2, 3, 4, 6, 8,]}[GRAIN]

for window in MA_WINDOWS:
    t0 = time.perf_counter()
    y_pred_ma = moving_average(train, valid, TARGET, window=window)
    fit_time = time.perf_counter() - t0
    evaluate_model(f"Moving average ({window}{GRAIN[0]})", y_pred_ma, fit_time=fit_time, category="Naive")

2026-08-22 22:16:25.014 | INFO     | src.evaluation.metrics:evaluate_predictions:326 - Moving average (2w) [Naive] | WAPE: 34.94% | WRMSSE: 1.4716 | MAE: 31.584 | RMSE: 84.156 | MAPE: 19757297607.69% | SMAPE: 46.22% | Bias: -8.47% | RMSLE: 1.1474 | TS: 38453.88 | SPEC: 4971.815 | MASE: 1.7419 | fit: 0.66s
2026-08-22 22:16:33.440 | INFO     | src.evaluation.metrics:evaluate_predictions:326 - Moving average (3w) [Naive] | WAPE: 34.51% | WRMSSE: 1.4579 | MAE: 31.197 | RMSE: 84.288 | MAPE: 20282627478.67% | SMAPE: 45.40% | Bias: -6.91% | RMSLE: 1.1294 | TS: 31738.62 | SPEC: 4822.582 | MASE: 1.7290 | fit: 0.61s
2026-08-22 22:16:41.530 | INFO     | src.evaluation.metrics:evaluate_predictions:326 - Moving average (4w) [Naive] | WAPE: 34.14% | WRMSSE: 1.4404 | MAE: 30.861 | RMSE: 82.798 | MAPE: 21220829160.89% | SMAPE: 45.10% | Bias: -6.64% | RMSLE: 1.1224 | TS: 30861.61 | SPEC: 4655.342 | MASE: 1.7088 | fit: 0.59s
2026-08-22 22:16:49.832 | INFO     | src.evaluation.metrics:evaluate_prediction

## Modelos estadísticos clásicos

Familias de forecasting estadístico clásico, ajustadas serie por serie (sin usar la matriz de features de ML). SARIMA es el modelo más citado en la literatura académica; ETS/Holt-Winters es rápido y robusto (base de benchmarks M4/M5); Theta ganó la M3 pese a su simplicidad; TBATS extiende ETS para estacionalidades múltiples; Prophet (Meta) es popular en industria por ser fácil de tunear.

In [16]:
from src.modeling import fit_sarima, fit_ets, fit_theta, fit_tbats, fit_prophet

### SARIMA

In [17]:
t0 = time.perf_counter()
y_pred_sarima = fit_sarima(train, valid, TARGET, seasonal_order=(1, 1, 1, SEASON_LENGTH))
fit_time = time.perf_counter() - t0

evaluate_model("SARIMA", y_pred_sarima, fit_time=fit_time, category="Statistical")

/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsforecast/arima.py:701: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(se * model["sigma2"])
/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsforecast/arima.py:701: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(se * model["sigma2"])
/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsforecast/arima.py:701: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(se * model["sigma2"])
/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsforecast/arima.py:701: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(se * model["sigma2"])
/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsforecast/arima.py:701: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(se * model["sigma2"])
/home/njimenez/

{'model': 'SARIMA',
 'category': 'Statistical',
 'wape': 22381458606994.58,
 'wrmsse': nan,
 'mae': 2040832336301055.2,
 'rmse': 2.5941488266875862e+17,
 'mape': nan,
 'smape': 0.5809533826430521,
 'bias': -9463923917072.46,
 'rmsle': nan,
 'tracking_signal': nan,
 'spec': nan,
 'mase': nan,
 'fit_time': 1028.3784632960014}

### ETS (Holt-Winters)

In [18]:
t0 = time.perf_counter()
y_pred_ets = fit_ets(train, valid, TARGET, seasonal_periods=SEASON_LENGTH)
fit_time = time.perf_counter() - t0

evaluate_model("ETS (Holt-Winters)", y_pred_ets, fit_time=fit_time, category="Statistical")

/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Opti

{'model': 'ETS (Holt-Winters)',
 'category': 'Statistical',
 'wape': 0.38893190172565567,
 'wrmsse': 1.6353766434416055,
 'mae': 35.161972557194986,
 'rmse': 93.23933787938098,
 'mape': 254328354.22217992,
 'smape': 0.5059597128217773,
 'bias': -0.016100638137681797,
 'rmsle': 1.1563780621848783,
 'tracking_signal': 6563.421421917226,
 'spec': 4225.98800172799,
 'mase': 1.9844563889138267,
 'fit_time': 225.82517712700064}

### Theta

In [19]:
t0 = time.perf_counter()
y_pred_theta = fit_theta(train, valid, TARGET, period=SEASON_LENGTH)
fit_time = time.perf_counter() - t0

evaluate_model("Theta", y_pred_theta, fit_time=fit_time, category="Statistical")

2026-08-22 22:38:43.303 | INFO     | src.evaluation.metrics:evaluate_predictions:326 -       Theta [Statistical] | WAPE: 34.41% | WRMSSE: 1.4473 | MAE: 31.106 | RMSE: 81.995 | MAPE: 21779305736.89% | SMAPE: 46.88% | Bias: -5.39% | RMSLE: 1.0910 | TS: 24816.39 | SPEC: 4196.240 | MASE: 1.7043 | fit: 17.00s


{'model': 'Theta',
 'category': 'Statistical',
 'wape': 0.3440719774524764,
 'wrmsse': 1.4473442681498319,
 'mae': 31.10634374605156,
 'rmse': 81.9948299188873,
 'mape': 217793057.36887354,
 'smape': 0.4688388710625243,
 'bias': -0.05385512971375739,
 'rmsle': 1.0909515620308374,
 'tracking_signal': 24816.386295324413,
 'spec': 4196.239756977053,
 'mase': 1.704258416698277,
 'fit_time': 16.998098283002037}

### TBATS

In [20]:
t0 = time.perf_counter()
y_pred_tbats = fit_tbats(train, valid, TARGET, season_length=(SEASON_LENGTH,))
fit_time = time.perf_counter() - t0

evaluate_model("TBATS", y_pred_tbats, fit_time=fit_time, category="Statistical")

/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsforecast/tbats.py:949: UserWarning: Data contains zero or negative values, disabling Box-Cox transformation.
  warnings.warn(
/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsforecast/tbats.py:949: UserWarning: Data contains zero or negative values, disabling Box-Cox transformation.
  warnings.warn(
/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsforecast/tbats.py:949: UserWarning: Data contains zero or negative values, disabling Box-Cox transformation.
  warnings.warn(
/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsforecast/tbats.py:949: UserWarning: Data contains zero or negative values, disabling Box-Cox transformation.
  warnings.warn(
/home/njimenez/workspace/magister/forecast-tfm/.venv/lib/python3.13/site-packages/statsforecast/tbats.py:949: UserWarning: Data contains zero or neg

{'model': 'TBATS',
 'category': 'Statistical',
 'wape': 0.5058440650467176,
 'wrmsse': 2.0259345729249296,
 'mae': 45.73159222598008,
 'rmse': 139.43134654920158,
 'mape': 287440531.59483993,
 'smape': 0.5969468594789791,
 'bias': -0.047808258420887324,
 'rmsle': 1.3616275668545341,
 'tracking_signal': 14984.664800633365,
 'spec': 8737.981804323776,
 'mase': 2.532053508134129,
 'fit_time': 92.38110545699965}

### Prophet

In [21]:
t0 = time.perf_counter()
y_pred_prophet = fit_prophet(train, valid, TARGET, weekly_seasonality=(GRAIN == "daily"))
fit_time = time.perf_counter() - t0

evaluate_model("Prophet", y_pred_prophet, fit_time=fit_time, category="Statistical")

22:40:38 - cmdstanpy - INFO - Chain [1] start processing
22:40:38 - cmdstanpy - INFO - Chain [1] start processing
22:40:38 - cmdstanpy - INFO - Chain [1] start processing
22:40:38 - cmdstanpy - INFO - Chain [1] start processing
22:40:38 - cmdstanpy - INFO - Chain [1] start processing
22:40:38 - cmdstanpy - INFO - Chain [1] start processing
22:40:38 - cmdstanpy - INFO - Chain [1] start processing
22:40:38 - cmdstanpy - INFO - Chain [1] start processing
22:40:38 - cmdstanpy - INFO - Chain [1] start processing
22:40:38 - cmdstanpy - INFO - Chain [1] start processing
22:40:38 - cmdstanpy - INFO - Chain [1] start processing
22:40:38 - cmdstanpy - INFO - Chain [1] start processing
22:40:38 - cmdstanpy - INFO - Chain [1] done processing
22:40:38 - cmdstanpy - INFO - Chain [1] done processing
22:40:38 - cmdstanpy - INFO - Chain [1] done processing
22:40:38 - cmdstanpy - INFO - Chain [1] done processing
22:40:38 - cmdstanpy - INFO - Chain [1] done processing
22:40:38 - cmdstanpy - INFO - Chain 

{'model': 'Prophet',
 'category': 'Statistical',
 'wape': 0.4194719492266324,
 'wrmsse': 1.7499802488942933,
 'mae': 37.922991407436406,
 'rmse': 95.27223855760386,
 'mape': 258875344.17230546,
 'smape': 0.535508036569128,
 'bias': -0.04196350591388567,
 'rmsle': 1.1509536158639906,
 'tracking_signal': 15860.965072637398,
 'spec': 7074.064441967375,
 'mase': 2.1796148717641985,
 'fit_time': 91.3587312769996}

## Modelos Machine Learning

In [22]:
from src.modeling import fit_lightgbm, fit_xgboost, fit_catboost, catboost_features, fit_histgb, fit_ridge

RANDOM_STATE = 42

### LightGBM

In [23]:
t0 = time.perf_counter()
lgbm_model = fit_lightgbm(X_train, y_train, CATEGORICAL_FEATURES, random_state=RANDOM_STATE)
fit_time = time.perf_counter() - t0

evaluate_model("LightGBM", lgbm_model.predict(X_valid), fit_time=fit_time, category="ML")

2026-08-22 22:42:36.358 | INFO     | src.evaluation.metrics:evaluate_predictions:326 -             LightGBM [ML] | WAPE: 16.48% | WRMSSE: 0.8297 | MAE: 14.899 | RMSE: 35.156 | MAPE: 1693474221.25% | SMAPE: 32.60% | Bias: -0.82% | RMSLE: 0.4810 | TS: 7843.47 | SPEC: 338.403 | MASE: 0.8928 | fit: 14.38s


{'model': 'LightGBM',
 'category': 'ML',
 'wape': 0.16480247819884025,
 'wrmsse': 0.8296594949975642,
 'mae': 14.89921549267219,
 'rmse': 35.15597022966511,
 'mape': 16934742.212463554,
 'smape': 0.3260166433219701,
 'bias': -0.008152880781266244,
 'rmsle': 0.48103510522107945,
 'tracking_signal': 7843.467866718627,
 'spec': 338.4029848397801,
 'mase': 0.8927563423842724,
 'fit_time': 14.379449530999409}

### XGBoost

In [24]:
t0 = time.perf_counter()
xgb_model = fit_xgboost(X_train, y_train, random_state=RANDOM_STATE)
fit_time = time.perf_counter() - t0

evaluate_model("XGBoost", xgb_model.predict(X_valid), fit_time=fit_time, category="ML")

2026-08-22 22:43:04.957 | INFO     | src.evaluation.metrics:evaluate_predictions:326 -              XGBoost [ML] | WAPE: 17.05% | WRMSSE: 0.8479 | MAE: 15.414 | RMSE: 39.321 | MAPE: 2295501570.48% | SMAPE: 31.25% | Bias: -3.82% | RMSLE: 0.4497 | TS: 35516.62 | SPEC: 690.399 | MASE: 0.9135 | fit: 20.44s


{'model': 'XGBoost',
 'category': 'ML',
 'wape': 0.17049729925883556,
 'wrmsse': 0.8478851989047402,
 'mae': 15.41406434138128,
 'rmse': 39.32085894856125,
 'mape': 22955015.704816714,
 'smape': 0.312525756893409,
 'bias': -0.03819339963724968,
 'rmsle': 0.4496598612581607,
 'tracking_signal': 35516.61611069686,
 'spec': 690.3985999311067,
 'mase': 0.913477982379305,
 'fit_time': 20.439363336998213}

### CatBoost

In [25]:
t0 = time.perf_counter()
catboost_model = fit_catboost(X_train, y_train, CATEGORICAL_FEATURES, random_state=RANDOM_STATE)
fit_time = time.perf_counter() - t0

X_valid_cb = catboost_features(X_valid, CATEGORICAL_FEATURES)

evaluate_model("CatBoost", catboost_model.predict(X_valid_cb), fit_time=fit_time, category="ML")

2026-08-22 22:49:53.580 | INFO     | src.evaluation.metrics:evaluate_predictions:326 -             CatBoost [ML] | WAPE: 16.76% | WRMSSE: 0.8443 | MAE: 15.153 | RMSE: 37.289 | MAPE: 1913367603.35% | SMAPE: 32.60% | Bias: -3.48% | RMSLE: 0.4528 | TS: 32936.48 | SPEC: 473.758 | MASE: 0.9242 | fit: 358.62s


{'model': 'CatBoost',
 'category': 'ML',
 'wape': 0.16761506522177252,
 'wrmsse': 0.844315921068611,
 'mae': 15.153491645582955,
 'rmse': 37.288879314270986,
 'mape': 19133676.03348914,
 'smape': 0.3260139428946458,
 'bias': -0.03482005232194871,
 'rmsle': 0.4528396015572408,
 'tracking_signal': 32936.47649294482,
 'spec': 473.7577467935608,
 'mase': 0.9241930561449603,
 'fit_time': 358.6183791700023}

### HistGradientBoosting (sklearn)

In [26]:
t0 = time.perf_counter()
hgb_model = fit_histgb(X_train, y_train, random_state=RANDOM_STATE)
fit_time = time.perf_counter() - t0

evaluate_model("HistGradientBoosting", hgb_model.predict(X_valid), fit_time=fit_time, category="ML")

ValueError: Categorical feature 'item_id' is expected to have a cardinality <= 255 but actually has a cardinality of 3049.

### Ridge (sklearn)

Modelo lineal de referencia; solo usa las features numéricas (no maneja categóricas de alta cardinalidad).

In [ ]:
t0 = time.perf_counter()
ridge_model = fit_ridge(X_train, y_train, NUMERICAL_FEATURES, random_state=RANDOM_STATE)
fit_time = time.perf_counter() - t0

evaluate_model("Ridge", ridge_model.predict(X_valid[NUMERICAL_FEATURES]), fit_time=fit_time, category="ML")

### Red neuronal (desde Colab)

Resultado entrenado aparte en `02_1_red_neuronal.ipynb` (Google Colab, con GPU). Se pega acá el JSON impreso al final de ese notebook (`evaluate_predictions` sobre el mismo set de validación) para incorporarlo a `model_results` y compararlo en la misma tabla.

In [ ]:
import json

# Pegar aca el JSON impreso al final de 02_1_red_neuronal.ipynb
nn_result_json = """
{
  "model": "Red neuronal (PyTorch)",
  "category": "Neural Network",
  "wape": 0.10028197523732978,
  "wrmsse": 0.6648106210840594,
  "mae": 56.451958003855964,
  "rmse": 101.11262700533668,
  "mape": 0.14361555839435278,
  "smape": 0.13450871897523384,
  "bias": -0.01973313975149749,
  "rmsle": 0.18986484887682267,
  "tracking_signal": 5027.640505260812,
  "spec": 24816.267370544832,
  "mase": 0.6779046862289563,
  "fit_time": 27.297044813000014
}
"""

nn_result = json.loads(nn_result_json)
if nn_result.get("model"):
    model_results.append(nn_result)
    print(f"Agregado: {nn_result['model']}")
else:
    print("Pega el JSON de 02_1_red_neuronal.ipynb en nn_result_json antes de correr esta celda")

## Resultados

In [ ]:
def plot_barh_by_model(results_df, x, y="model", hue=None, title="", figsize=(8, 6)):
    """Barplot horizontal genérico para comparar modelos por una métrica."""
    df_sorted = results_df.sort_values(x, ascending=True)
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(df_sorted, x=x, y=y, hue=hue, order=df_sorted[y], ax=ax)
    ax.set_title(title, loc="left", pad=20)
    ax.set_xlabel("")
    ax.set_ylabel("")
    sns.despine()
    if hue:
        ax.legend(title=hue, loc="upper right")
    plt.tight_layout()
    plt.show()

In [ ]:
results_df = pd.DataFrame(model_results).sort_values("wrmsse")
results_df

### Métricas por serie para todos los modelos

`build_series_metrics` da WAPE/bias/RMSLE/tracking signal/WRMSSE/MASE/SPEC por serie para un modelo. `build_all_series_metrics` lo corre para todos los modelos de `predictions_valid` (mismo set de validación) y los concatena en un único DataFrame con columna `model`, para comparar múltiples métricas a nivel de serie entre modelos (no solo el agregado de `results_df`).

In [ ]:
from src.evaluation import build_all_series_metrics

series_metrics_df = build_all_series_metrics(train, valid, y_valid, predictions_valid, target_col=TARGET)
series_metrics_df.sort_values(["model", "gross_sales"], ascending=[True, False]).head(20)

In [ ]:
for col in ['wape', 'wrmsse', 'mae', 'rmse', 'mape', 'smape', 'bias', 'rmsle', 'tracking_signal', 'spec', 'mase', 'fit_time']:
    plot_barh_by_model(results_df, hue="category", x=col, y="model", title=f"{col} por modelo")

In [ ]:
for col in ['wrmsse','wape','rmse', 'fit_time']:
    plot_barh_by_model(results_df.query('category == "ML"'), x=col, y="model", title=f"{col} por modelo")

# Selección de features

El resto del notebook usa LightGBM (velocidad, soporte nativo de categóricas, compatible con SHAP), así que la selección se hace sobre `lgbm_model`. Se comparan dos criterios de importancia (gain del árbol vs. permutation importance sobre validación) y luego se aplica una eliminación hacia atrás (backward elimination): se van descartando las features menos importantes mientras el WRMSSE de validación no empeore más allá de una tolerancia.

## Importancia por ganancia (gain)

In [ ]:
importance_gain = (
    pd.Series(lgbm_model.booster_.feature_importance(importance_type="gain"), index=FEATURES)
    .sort_values(ascending=False)
)
importance_gain.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
importance_gain.head(30).sort_values().plot.barh(ax=ax)
ax.set_title("Importancia por ganancia (LightGBM, top 30)", loc="left")
sns.despine()
plt.tight_layout()
plt.show()

## Permutation importance

El gain puede sobreestimar features de alta cardinalidad o muy usadas para splits sin aportar demasiado al error final. La permutation importance mide directamente cuánto empeora el WAPE en validación al mezclar (shuffle) cada columna, así que es un criterio más fiel al desempeño real del modelo.

In [ ]:
from src.modeling import compute_permutation_importance

importance_perm = compute_permutation_importance(
    lgbm_model, X_valid, y_valid, FEATURES,
    sample_size=15_000, n_repeats=10, random_state=RANDOM_STATE,
)
importance_perm.head(20)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
importance_perm.head(30).sort_values().plot.barh(ax=ax)
ax.set_title("Permutation importance (RMSE, top 30)", loc="left")
sns.despine()
plt.tight_layout()
plt.show()

## Eliminación hacia atrás (backward elimination)

Se parte del ranking de permutation importance (de menor a mayor) y se intenta eliminar cada feature: si reentrenar sin ella no empeora el WRMSSE de validación más allá de `TOLERANCE`, se descarta definitivamente.

In [ ]:
from src.modeling import backward_feature_selection

selected_features, final_wrmsse, log_df = backward_feature_selection(
    X_train, y_train, X_valid, train, valid,
    FEATURES, CATEGORICAL_FEATURES, importance_perm,
    tolerance=0.001, random_state=RANDOM_STATE,
)


In [ ]:
baseline_wrmsse = log_df.loc[log_df["removed"].isna(), "wrmsse"].iloc[0]

# Trayectoria real: baseline + solo las eliminaciones aceptadas, en orden
trajectory = log_df[log_df["accepted"]].sort_values("n_features", ascending=False)
baseline_wrmsse = trajectory["wrmsse"].iloc[0]

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(trajectory, x="n_features", y="wrmsse", marker="o", ax=ax)
ax.invert_xaxis()
ax.axhline(baseline_wrmsse, color="grey", linestyle="--", linewidth=1, label="baseline (todas las features)")
ax.set_title("WRMSSE de validación durante la eliminación hacia atrás", loc="left")
ax.set_xlabel("Cantidad de features")
ax.set_ylabel("WRMSSE")
ax.legend(loc="upper right")
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
baseline_wrmsse = log_df.loc[log_df["removed"].isna(), "wrmsse"].iloc[0]

# Trayectoria real: baseline + solo las eliminaciones aceptadas, en orden
trajectory = log_df[log_df["accepted"]].sort_values("n_features", ascending=False)
baseline_wrmsse = trajectory["wrmsse"].iloc[0]

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(trajectory, x="n_features", y="wrmsse", marker="o", ax=ax)
ax.invert_xaxis()
ax.axhline(baseline_wrmsse, color="grey", linestyle="--", linewidth=1, label="baseline (todas las features)")
ax.set_title("WRMSSE de validación durante la eliminación hacia atrás", loc="left")
ax.set_xlabel("Cantidad de features")
ax.set_ylabel("WRMSSE")
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

## Features seleccionadas

Se fija `FEATURES` (y las variables derivadas `X_train`/`X_valid`/`X_test`/`CATEGORICAL_FEATURES`/`NUMERICAL_FEATURES`) al subconjunto seleccionado, para que el resto del notebook (modelo simple, SHAP, tuning con Optuna, modelo final) entrene sobre las features filtradas.

In [ ]:
print(f"Features descartadas ({len(FEATURES) - len(selected_features)}): "
      f"{sorted(set(FEATURES) - set(selected_features))}")

FEATURES = selected_features
CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if c in FEATURES]
NUMERICAL_FEATURES = [c for c in NUMERICAL_FEATURES if c in FEATURES]

X_train = X_train[FEATURES]
X_valid = X_valid[FEATURES]
X_test = X_test[FEATURES]

print(f"Features finales: {len(FEATURES)} ({len(CATEGORICAL_FEATURES)} categóricas)")

# Modelo simple

In [ ]:
from lightgbm import LGBMRegressor
import lightgbm as lgb
from src.evaluation import wape_metric, make_wrmsse_metric

wrmsse_metric = make_wrmsse_metric(train, valid)


In [ ]:
model = LGBMRegressor(
    objective="rmse",
)

t0 = time.perf_counter()
model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric=[wrmsse_metric,],
    callbacks=[
        lgb.early_stopping(100, first_metric_only=True),
        lgb.log_evaluation(10),
    ],
)
fit_time = time.perf_counter() - t0

evaluate_model("LightGBM (selected features)", model.predict(X_valid), fit_time=fit_time, category="ML")

## Explicación modelo

In [ ]:
import shap

explainer = shap.TreeExplainer(model)

X_shap = X_test.sample(n=min(2000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_shap)

In [ ]:
shap_df = pd.DataFrame(shap_values, columns=X_shap.columns, index=X_shap.index)

importance_df = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

print(importance_df)

### Importancia global de features

In [ ]:
shap.summary_plot(shap_values, X_shap, plot_type="bar")

In [ ]:
shap.summary_plot(shap_values, X_shap)

### Dependencia por feature

In [ ]:
top_feature = X_shap.columns[np.argsort(-np.abs(shap_values).mean(axis=0))[0]]
print(f"Feature más importante: {top_feature}")

shap.dependence_plot(top_feature, shap_values, X_shap, interaction_index=None)

# Predicciones

In [ ]:
from src.evaluation import explain_prediction


In [ ]:
from src.evaluation import plot_forecast


In [ ]:
from src.evaluation import analizar_prediccion as _analizar_prediccion

def analizar_prediccion(agg_id, date=None, max_display=10):
    return _analizar_prediccion(
        test, X_test, df_pred, TARGET, agg_id,
        date=date, max_display=max_display, explainer=explainer,
    )


In [ ]:
from src.evaluation.metrics import build_predictions_report

metrics_test, df_pred = build_predictions_report(train, test, y_test, model.predict(X_test), target_col=TARGET)

print(f"Test WAPE: {metrics_test['wape']:.2%}")
print(f"Test WRMSSE: {metrics_test['wrmsse']:.4f}")

In [ ]:
from src.evaluation.metrics import build_series_metrics

df_metrics = build_series_metrics(train, df_pred, target_col=TARGET, weight_level=["date"])
df_metrics.head(10)

In [ ]:
df_pred.query('agg_id == "FOODS_3_FOODS_CA_3_CA"').nlargest(10, 'wape')

In [ ]:
analizar_prediccion('FOODS_3_FOODS_CA_3_CA', '2016-05-16',)

# Modelo optimizado

In [ ]:
import optuna
from optuna.integration import LightGBMPruningCallback
import lightgbm as lgb
import numpy as np

In [ ]:
import lightgbm as lgb
import numpy as np
import optuna
from optuna.integration import LightGBMPruningCallback

In [ ]:
def objective(trial):
    model = lgb.LGBMRegressor(
        objective="rmse",
        learning_rate=trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
        n_estimators=1500,
        num_leaves=trial.suggest_int("num_leaves", 31, 255),
        max_depth=trial.suggest_int("max_depth", 5, 10),
        min_child_samples=trial.suggest_int("min_child_samples", 20, 200),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 5, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 5, log=True),
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="rmse",
        categorical_feature=CATEGORICAL_FEATURES,
        callbacks=[
            lgb.early_stopping(30, first_metric_only=True, verbose=False),
            LightGBMPruningCallback(trial, "rmse"),
        ],
    )
    y_pred = model.predict(X_valid, num_iteration=model.best_iteration_)
    _, final_wrmsse, _ = wrmsse_metric(y_valid, y_pred)
    return final_wrmsse

In [ ]:
study = optuna.create_study(
    study_name=f"study_{LEVEL}", 
    direction="minimize",
    storage="sqlite:///../artifacts/optuna_study.db",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(
        seed=42,
       #multivariate=True,
        #group=True,
    ),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10, n_startup_trials=5),
)

study.optimize(
    objective,
    n_trials=1_000,
    timeout=60 * 3,
    show_progress_bar=True,
)

best_trial = study.best_trial
print(f"Mejor WRMSSE: {best_trial.value:.4f}")
print("Mejores hiperparámetros:", best_trial.params)

# Modelo final

In [ ]:
model_params = best_trial.params

final_model = lgb.LGBMRegressor(
    objective="rmse",
    n_estimators=1000,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
    subsample_freq=1,
    **model_params,
)

t0 = time.perf_counter()
final_model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric=wrmsse_metric,
    categorical_feature=CATEGORICAL_FEATURES,
    callbacks=[
        lgb.early_stopping(50, first_metric_only=True),
        lgb.log_evaluation(10),
    ],
)
fit_time = time.perf_counter() - t0

evaluate_model("LightGBM (selected features | Optuna)", final_model.predict(X_valid), fit_time=fit_time, category="ML")
final_model.booster_.save_model(f"../artifacts/{LEVEL}_final_model.txt")

## Resultados finales

In [ ]:
results_df = pd.DataFrame(model_results).sort_values("wrmsse").query('wape <= 0.15')
results_df

In [ ]:
for col in ['wape', 'wrmsse', 'mae', 'rmse', 'mape', 'smape', 'bias', 'rmsle', 'tracking_signal', 'spec', 'mase', 'fit_time']:
    plot_barh_by_model(results_df, hue="category", x=col, y="model", title=f"{col} por modelo")

# Exploración modelo final

In [ ]:
metrics_test_final, df_pred = build_predictions_report(train, test, y_test, final_model.predict(X_test), target_col=TARGET)

print(f"Test WAPE: {metrics_test_final['wape']:.2%}")
print(f"Test WRMSSE: {metrics_test_final['wrmsse']:.4f}")

In [ ]:
from src.evaluation.metrics import build_series_metrics

df_metrics = build_series_metrics(train, df_pred, target_col=TARGET, weight_level=["date"])
df_metrics.head(10)

In [ ]:
df_pred.query('agg_id == "FOODS_3_FOODS_CA_3_CA"').nlargest(10, 'wape')

In [ ]:
analizar_prediccion('FOODS_3_FOODS_CA_3_CA', '2016-05-15',)

# Exportar

In [ ]:
import joblib

feature_importance = (
    pd.Series(final_model.booster_.feature_importance(importance_type="gain"), index=FEATURES)
    .sort_values(ascending=False)
)

artifact = {
    # modelo y datos de trabajo
    "level": LEVEL,
    "metrics_results": results_df,
    "model": final_model,
    "model_params": model_params,
    "X_train": X_train,
    "y_train": y_train,
    "X_valid": X_valid,
    "y_valid": y_valid,
    "X_test": X_test,
    "y_test": y_test,
    "train": train,
    "valid": valid,
    "test": test,
    # features y columnas
    "features": FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "numerical_features": NUMERICAL_FEATURES,
    "id_cols": ID_COLS,
    "leaky_cols": LEAKY_COLS,
    "target": TARGET,
    "feature_importance": feature_importance,
    # predicciones y métricas del modelo final (test)
    "df_pred": df_pred,
    "df_metrics": df_metrics,
    # comparación de todos los modelos probados (validación)
    "model_results": model_results,
    "results_df": results_df,
    #"series_metrics_valid": series_metrics_df,
    # metadatos
    "wape_valid": model_results[-1]["wape"],
    "wrmsse_valid": model_results[-1]["wrmsse"],
    "wape_test": metrics_test_final["wape"],
    "wrmsse_test": metrics_test_final["wrmsse"],
    "train_start": str(first_date.date()),
    "valid_start": str(valid_start.date()),
    "test_start": str(test_start.date()),
}

artifact_path = f"../artifacts/{LEVEL}_artifact.pkl"
joblib.dump(artifact, artifact_path)
print(f"Artifact guardado en {artifact_path}")